In [1]:
#!pip install spotipy

In [2]:
import spotipy 
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv
import os
import pandas as pd


In [11]:
load_dotenv()
client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")
redirect_uri = os.getenv("SPOTIFY_REDIRECT_URI")

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id = client_id,
    client_secret = client_secret,
    redirect_uri = redirect_uri,
    scope='user-library-read playlist-read-private playlist-read-collaborative'
))

### Extracting Liked Songs

In [6]:
def get_liked_songs(sp):
    results = []
    offset = 0
    limit = 50

    while True:
        response = sp.current_user_saved_tracks(limit=limit, offset=offset)
        items = response['items']

        if not items:
            break
        
        results.extend(items)
        offset += limit

    return results

liked_songs = get_liked_songs(sp)
print(len(liked_songs))

liked_songs_df = pd.DataFrame(liked_songs)
liked_songs_df.to_csv('data/liked_songs_raw.csv')

2120


### Track Metadata Extraction

In [8]:
def parse_tracks(tracks):
    data = []

    for item in tracks:
        track = item['track']
        data.append({
            'track_name': track['name'],
            'artist': track['artists'][0]['name'],
            'track_id': track['id'],
            'album': track['album']['name'],
            'release_date': track['album']['release_date']
        })
    
    return pd.DataFrame(data)

track_metadata = parse_tracks(liked_songs)
track_metadata.head()

track_metadata = pd.DataFrame(track_metadata)
track_metadata.to_csv('data/track_metadata_raw.csv')

### Recommendation Set Extraction

In [25]:
def get_recommendation_sets(sp, seed_track):
   tracks = []
   offset = 0 
   limit = 50
   while True:
      response = sp.recommendations(seed_track, limit=limit, offset=offset)
      items = response['items']
      if not items:
         break
      for item in items:
         track = item['track']
         if track:
            tracks.append({
                  'track_name': track['name'],
                  'artist': track['artists'][0]['name'],
                  'track_id': track['id'],
                  'album': track['album']['name'],
                  'release_date': track['album']['release_date']
            })
      offset += limit
   return tracks

seeds = {
   
}


for seed_name, seed_id in seeds.items():
   tracks = get_recommendation_sets(sp, seed_id)
   filename = f"{seed_name.replace(' ', '_')}_recommendationSet.csv"
   pd.DateFrame(tracks).to_csv(filename, index=False)
   print(f"Saved {len(tracks)} tracks to {filename}")